### Word2Vec

In [1]:
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [2]:
df=data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


### Preprocessing and Cleaning

In [3]:
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

In [4]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [5]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [6]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

# Load stopwords ONCE
stop_words = set(stopwords.words('english'))

corpus = []

for review in df['reviewText']:

    # Convert to string
    review = str(review)

    # Remove URLs
    review = re.sub(
        r'https?://\S+|www\.\S+',
        '',
        review
    )

    # Remove HTML tags
    review = BeautifulSoup(review, 'html.parser').get_text()

    # Remove special characters
    review = re.sub('[^a-zA-Z0-9-]+', ' ', review)

    # Lowercase
    review = review.lower()

    # Remove stopwords
    words = [
        word for word in review.split()
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Join
    review = ' '.join(words)

    # Remove extra spaces
    review = ' '.join(review.split())

    corpus.append(review)


In [7]:
df['reviewText']=corpus

In [8]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting conclud...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [9]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [10]:
# Converts the reviewText into tokenized sentences, so that it can be given to Word2Vec
words = []

for paragraph in df['reviewText']:
    sent_token = sent_tokenize(paragraph)

    for sentence in sent_token:
        words.append(simple_preprocess(sentence))

In [11]:
# Training word2vec from scratch
import gensim
model=gensim.models.Word2Vec(words)

In [12]:
model.wv.index_to_key

['book',
 'story',
 'read',
 'one',
 'character',
 'like',
 'good',
 'would',
 'love',
 'really',
 'time',
 'get',
 'author',
 'series',
 'well',
 'reading',
 'much',
 'first',
 'even',
 'short',
 'way',
 'great',
 'could',
 'know',
 'sex',
 'make',
 'little',
 'two',
 'thing',
 'want',
 'plot',
 'think',
 'romance',
 'also',
 'find',
 'end',
 'life',
 'see',
 'scene',
 'go',
 'enjoyed',
 'never',
 'written',
 'woman',
 'take',
 'kindle',
 'many',
 'lot',
 'year',
 'work',
 'say',
 'thought',
 'interesting',
 'bit',
 'going',
 'found',
 'novel',
 'writing',
 'give',
 'another',
 'liked',
 'loved',
 'better',
 'man',
 'hot',
 'got',
 'feel',
 'come',
 'still',
 'star',
 'review',
 'though',
 'back',
 'enough',
 'reader',
 'people',
 'friend',
 'made',
 'something',
 'page',
 'part',
 'world',
 'bad',
 'free',
 'need',
 'keep',
 'new',
 'relationship',
 'enjoy',
 'together',
 'next',
 'recommend',
 'start',
 'felt',
 'best',
 'word',
 'put',
 'guy',
 'main',
 'however',
 'looking',
 'day

In [13]:
model.corpus_count

12000

In [14]:
model.wv.similar_by_word('good')

[('decent', 0.8629593849182129),
 ('great', 0.8376079797744751),
 ('nice', 0.8130007386207581),
 ('okay', 0.8099492192268372),
 ('satisfying', 0.8064167499542236),
 ('usually', 0.806216299533844),
 ('amazing', 0.8062124252319336),
 ('ok', 0.8055790662765503),
 ('alot', 0.8024327158927917),
 ('overall', 0.7962294816970825)]

In [15]:
# Applying avg_word2vec
import numpy as np

def avg_word2vec(doc):
  # Keep only known words
  sent = [word for word in doc if word in model.wv.index_to_key]

  # Handle empty document
  if len(sent) == 0:
    return np.zeros(model.vector_size)

  # Average word vectors
  return np.mean([model.wv[word] for word in sent], axis=0)

In [16]:
# Apply for the entire sentences
X=[]
for i in range(len(words)):
  X.append(avg_word2vec(words[i]))

In [17]:
X_new = np.array(X)

In [18]:
X_new.shape

(12000, 100)

In [ ]:
# Dependent Features
y = df['rating'].values
# If you already converted the ratings into binary values, then you don't need get_dummies() at all

In [20]:
y

array([1, 1, 1, ..., 1, 0, 1], shape=(12000,))

In [21]:
# Independent feature
# Converting vectors and dimensions into rows and columns
df = pd.DataFrame(X_new)

In [22]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.137631,0.301688,0.147085,0.064531,0.062885,-0.485979,0.127379,0.576724,-0.345531,-0.107846,...,0.355889,0.083428,-0.013309,0.160598,0.616677,0.328456,0.204552,-0.261392,-0.042007,-0.026641
1,-0.249870,0.169632,-0.162111,0.227867,-0.137714,-0.612084,0.563810,0.588352,-0.703625,-0.215723,...,0.587489,0.046775,-0.013559,-0.115468,0.642905,0.235853,0.310920,-0.235264,-0.035510,-0.092756
2,-0.180751,0.290766,-0.073855,0.072915,-0.026349,-0.541952,0.357874,0.619110,-0.410324,-0.038304,...,0.432618,0.067807,0.171322,0.037448,0.643425,0.230745,0.244863,-0.233626,-0.083096,0.073502
3,0.022321,0.252693,-0.211697,0.040665,0.050879,-0.512984,0.376378,0.683365,-0.269665,-0.108283,...,0.388726,-0.082476,0.206206,0.081191,0.596574,-0.029713,0.373120,-0.073340,-0.106151,0.165157
4,-0.175962,0.351993,0.317340,0.208738,-0.064674,-0.620663,0.347914,0.572066,-0.478042,-0.308298,...,0.657306,0.537261,0.131394,-0.253151,0.588312,0.292420,0.070885,-0.467900,0.280312,-0.007561


In [23]:
X=df

In [24]:
X.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.137631,0.301688,0.147085,0.064531,0.062885,-0.485979,0.127379,0.576724,-0.345531,-0.107846,...,0.355889,0.083428,-0.013309,0.160598,0.616677,0.328456,0.204552,-0.261392,-0.042007,-0.026641
1,-0.249870,0.169632,-0.162111,0.227867,-0.137714,-0.612084,0.563810,0.588352,-0.703625,-0.215723,...,0.587489,0.046775,-0.013559,-0.115468,0.642905,0.235853,0.310920,-0.235264,-0.035510,-0.092756
2,-0.180751,0.290766,-0.073855,0.072915,-0.026349,-0.541952,0.357874,0.619110,-0.410324,-0.038304,...,0.432618,0.067807,0.171322,0.037448,0.643425,0.230745,0.244863,-0.233626,-0.083096,0.073502
3,0.022321,0.252693,-0.211697,0.040665,0.050879,-0.512984,0.376378,0.683365,-0.269665,-0.108283,...,0.388726,-0.082476,0.206206,0.081191,0.596574,-0.029713,0.373120,-0.073340,-0.106151,0.165157
4,-0.175962,0.351993,0.317340,0.208738,-0.064674,-0.620663,0.347914,0.572066,-0.478042,-0.308298,...,0.657306,0.537261,0.131394,-0.253151,0.588312,0.292420,0.070885,-0.467900,0.280312,-0.007561


In [25]:
y

array([1, 1, 1, ..., 1, 0, 1], shape=(12000,))

In [26]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [27]:
from sklearn.ensemble import RandomForestClassifier
classifier=RandomForestClassifier()

In [28]:
classifier.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [29]:
y_pred=classifier.predict(X_test)

In [30]:
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))

0.7679166666666667


In [31]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.58      0.63       801
           1       0.81      0.86      0.83      1599

    accuracy                           0.77      2400
   macro avg       0.74      0.72      0.73      2400
weighted avg       0.76      0.77      0.76      2400

